---
execute:
  skip: true
authors:
- name: Steffen Bollmann
- name: "Mich\xE8le Masson-Trottier"
doi: 10.5281/zenodo.19043247
github: https://github.com/neurodesk/neurodeskedu/blob/main/books/tutorials/phase_processing/unwrapping.ipynb
edit_url: https://github.com/neurodesk/neurodeskedu/edit/main/books/tutorials/phase_processing/unwrapping.ipynb
downloads:
- url: /edu/_sources/tutorials/phase_processing/unwrapping.ipynb
  title: Download source notebook
  filename: unwrapping.ipynb
  static: false
---
# MRI Phase Unwrapping with ROMEO

:::{admonition} Unreviewed
:class: nd-review-badge nd-review-badge--unreviewed

[review issue](<https://github.com/neurodesk/neurodeskedu-reviews/issues/54>)
:::

:::{dropdown} Run this notebook
:class: nd-launch

- [Run in Australia](<https://play.neurodesk.cloud.edu.au/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/tutorials/phase_processing/unwrapping.ipynb&branch=main>)
- [Run in Europe](<https://play-europe.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/tutorials/phase_processing/unwrapping.ipynb&branch=main>)
- [Run in America](<https://play-america.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/tutorials/phase_processing/unwrapping.ipynb&branch=main>)
- [Run in Global Server 1](<https://play-europe.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/tutorials/phase_processing/unwrapping.ipynb&branch=main>)
- [Run in Global Server 2](<https://edu.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/tutorials/phase_processing/unwrapping.ipynb&branch=main>)
:::



**Author**: Steffen Bollmann, Michèle Masson-Trottier

The University of Queensland<br>
<div style="line-height: 2;">
<a href="https://github.com/stebo85"><img src="https://img.shields.io/badge/-Steffen_Bollmann-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-2909-0906"><img src="https://img.shields.io/badge/ORCID-0000--0002--2909--0906-green?logo=orcid" alt="ORCID"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 16/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>

## Purpose

MRI phase data are inherently **wrapped** — phase values outside the range $[-\pi, \pi]$ "wrap around", creating artificial discontinuities. **Phase unwrapping** recovers the true, continuous phase signal, which is essential for downstream analyses such as quantitative susceptibility mapping (QSM) and field mapping.

This tutorial demonstrates how to:

1. Download a demo dataset with magnitude and phase images
2. Prepare the data for unwrapping
3. Run **ROMEO** (Robust phase unwrapping) to recover continuous phase

:::{admonition} Learning Objectives
:class: tip
By the end of this tutorial you will be able to:
- Explain why MRI phase data need unwrapping
- Use ROMEO for robust, magnitude-guided phase unwrapping
- Interpret the unwrapped phase output
:::


## Citation and Resources

### Tools used in this workflow

__ROMEO__
: Dymerska, B., Eckstein, K., et al. (2021). Phase unwrapping with a rapid opensource minimum spanning tree algorithm (ROMEO). *Magnetic Resonance in Medicine*, 85(4), 2294–2308. [https://doi.org/10.1002/mrm.28563](https://doi.org/10.1002/mrm.28563)

### Educational resources

- [ROMEO on GitHub](https://github.com/korbinian90/ROMEO.jl)
- [Neurodesk documentation](https://neurodesk.org)


## Prerequisites

:::{admonition} Before you begin
:class: warning
Make sure you have access to a running Neurodesk instance. See [Getting Set Up with Neurodesk](https://neurodesk.org/getting-started/) for instructions.
:::

- [x] A running Neurodesk environment
- [ ] Familiarity with basic terminal commands
- [ ] No additional installation required — ROMEO is pre-installed in Neurodesk


## Load software tools and download demo data

We use the same demo dataset as the [SWI tutorial](swi.ipynb). Open a terminal in Neurodesktop and run the following commands to download the data, then copy and prepare the magnitude and phase files for ROMEO.

:::{tip}
To paste in Neurodesktop, best to use the right click on your mouse!
:::

![The terminal in Neurodesktop](/static/tutorials/phase_processing/swi/neurodesktop-terminal.png)
*The terminal in Neurodesktop.*

```bash
pip install osfclient
cd ~/neurodesktop-storage/
osf -p ru43c fetch -f 01_bids.zip ~/neurodesktop-storage/swi-demo/01_bids.zip

unzip -o ~/neurodesktop-storage/swi-demo/01_bids.zip -d ~/neurodesktop-storage/swi-demo/
```

Now copy the magnitude and phase files into a dedicated working directory:

```bash
mkdir -p ~/neurodesktop-storage/romeo-demo/

cp ~/neurodesktop-storage/swi-demo/01_bids/sub-170705134431std1312211075243167001/ses-1/anat/sub-170705134431std1312211075243167001_ses-1_run-1_part-phase_T2starw.nii \
   ~/neurodesktop-storage/romeo-demo/phase.nii

cp ~/neurodesktop-storage/swi-demo/01_bids/sub-170705134431std1312211075243167001/ses-1/anat/sub-170705134431std1312211075243167001_ses-1_run-1_part-mag_T2starw.nii \
   ~/neurodesktop-storage/romeo-demo/mag.nii
```

:::{note}
The files are already uncompressed `.nii` format, which is what ROMEO expects. No need to gunzip.
:::

![Terminal output after downloading and preparing data](/static/tutorials/phase_processing/unwrapping/terminal_download.png)
*Terminal output after downloading and preparing the data.*

## Run ROMEO for phase unwrapping

```{figure} /static/tutorials/phase_processing/unwrapping/open-romeo.png
:label: fig-open-romeo
:class: hidden

Opening ROMEO from the Neurodesk application menu.
```

Open the [ROMEO tool from the Neurodesk application menu](#fig-open-romeo), then run the following command in the ROMEO container terminal:

:::{note}
The `-k nomask` flag disables brain masking so all voxels are unwrapped. For most applications you would want to use a brain mask — see the [ROMEO documentation](https://github.com/korbinian90/ROMEO.jl) for options.
:::

```bash
romeo \
  -p ~/neurodesktop-storage/romeo-demo/phase.nii \
  -m ~/neurodesktop-storage/romeo-demo/mag.nii \
  -k nomask \
  -o ~/neurodesktop-storage/romeo-demo/
```

![Terminal output after running ROMEO](/static/tutorials/phase_processing/unwrapping/terminal_romeo.png)
*Terminal output after running ROMEO.*

## Inspect the results

The unwrapped phase image is saved in the output directory. You can verify it was created in the LXTerminal:

```bash
ls ~/neurodesktop-storage/romeo-demo/
```

```{figure} /static/tutorials/phase_processing/swi/open-itksnap.png
:label: fig-unwrap-open-itksnap
:class: hidden

Opening ITK-SNAP from the Neurodesk Visualisation menu.
```

Open [ITK-SNAP from the Neurodesk Visualisation menu](#fig-unwrap-open-itksnap) and load the original wrapped phase (`phase.nii`) alongside the unwrapped output to compare.

:::{tip}
Look for the removal of the sharp $2\pi$ jumps that were visible in the wrapped phase. The unwrapped phase should show smooth, continuous variation across the brain.
:::

![ROMEO unwrapping result — wrapped vs unwrapped phase](/static/tutorials/phase_processing/unwrapping/romeo_result.png)
*Comparison of wrapped (left) and unwrapped (right) phase in ITK-SNAP.*

## Summary

In this tutorial you:

1. Downloaded and prepared magnitude and phase MRI data
2. Used ROMEO to perform robust phase unwrapping
3. Compared the wrapped and unwrapped phase images

:::{seealso}
- [SWI tutorial](swi.ipynb) — for susceptibility-weighted imaging using the same dataset
- [QSM tutorial](qsm.ipynb) — for quantitative susceptibility mapping with QSMxT
:::